# 🧠 MindGuard — Train BERT Emotion Classifier

This notebook trains a **DistilBERT** model for **multi-label emotion classification** using Google's [GoEmotions](https://github.com/google-research/google-research/tree/master/goemotions) dataset (58K Reddit comments, 28 emotion labels).

### Before you start
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Run all cells in order
3. Download the `bert_emotion_model.zip` at the end
4. Extract it into your project's `models/bert_emotion/best_model/` folder

**Estimated time:** ~20–40 minutes on T4 GPU

## 1. Install Dependencies

In [ ]:
!pip install -q torch transformers datasets scikit-learn accelerate

## 2. Check GPU Availability

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("⚠️ No GPU detected! Training will be very slow.")
    print("Go to Runtime → Change runtime type → T4 GPU")

## 3. Configuration

In [ ]:
import random
import numpy as np

# ---------- You can tweak these ----------
MODEL_NAME = "distilbert-base-uncased"   # Base model to fine-tune
MAX_LENGTH = 128                          # Max tokens per input
LEARNING_RATE = 2e-5
BATCH_SIZE = 16                           # Reduce to 8 if you get OOM errors
EPOCHS = 3
SEED = 42
OUTPUT_DIR = "./bert_emotion_model"       # Where the trained model is saved
# -----------------------------------------

# 28 GoEmotions labels
LABELS = [
    "admiration", "amusement", "anger", "annoyance", "approval", "caring",
    "confusion", "curiosity", "desire", "disappointment", "disapproval",
    "disgust", "embarrassment", "excitement", "fear", "gratitude", "grief",
    "joy", "love", "nervousness", "optimism", "pride", "realization",
    "relief", "remorse", "sadness", "surprise", "neutral",
]
NUM_LABELS = len(LABELS)

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Model: {MODEL_NAME}")
print(f"Labels: {NUM_LABELS}")
print(f"Epochs: {EPOCHS}, Batch size: {BATCH_SIZE}, LR: {LEARNING_RATE}")

## 4. Load GoEmotions Dataset

In [ ]:
from datasets import load_dataset, Sequence, Value

dataset = load_dataset("google-research-datasets/go_emotions", "simplified")

print(f"Train:      {len(dataset['train']):,} examples")
print(f"Validation: {len(dataset['validation']):,} examples")
print(f"Test:       {len(dataset['test']):,} examples")
print(f"\nSample: {dataset['train'][0]}")

## 5. Tokenize & Prepare Labels

GoEmotions stores labels as a list of integer IDs (e.g. `[3, 14]` = annoyance + fear).  
We convert these to **multi-hot vectors** (e.g. `[0, 0, 0, 1, 0, ..., 1, 0, ...]`) for multi-label classification.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def labels_to_multihot(label_ids):
    """Convert list of label indices to a multi-hot float vector."""
    vector = [0.0] * NUM_LABELS
    for lid in label_ids:
        vector[lid] = 1.0
    return vector

def preprocess(batch):
    encoded = tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)
    encoded["labels"] = [labels_to_multihot(ids) for ids in batch["labels"]]
    return encoded

encoded_dataset = dataset.map(preprocess, batched=True, remove_columns=dataset["train"].column_names)
encoded_dataset = encoded_dataset.cast_column("labels", Sequence(feature=Value(dtype="float32")))

print("✅ Tokenization complete")
print(f"Features: {list(encoded_dataset['train'].features.keys())}")

## 6. Load Model

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
    id2label={i: label for i, label in enumerate(LABELS)},
    label2id={label: i for i, label in enumerate(LABELS)},
    ignore_mismatched_sizes=True,
)

total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable:,}")

## 7. Define Metrics

In [ ]:
from sklearn.metrics import f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Sigmoid to convert logits → probabilities
    probs = 1.0 / (1.0 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)
    labels = labels.astype(int)

    micro_f1 = f1_score(labels, preds, average="micro", zero_division=0)
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
    return {"micro_f1": float(micro_f1), "macro_f1": float(macro_f1)}

print("✅ Metrics function ready")

## 8. Train! 🚀

This is the main training loop. On a T4 GPU, expect:
- ~7 min per epoch
- ~20-25 min total for 3 epochs

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

training_args = TrainingArguments(
    output_dir="./checkpoints",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=32,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_micro_f1",
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=50,
    seed=SEED,
    report_to=[],  # disable wandb/mlflow in Colab
    fp16=torch.cuda.is_available(),  # mixed precision for faster training
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

print("Starting training...")
train_result = trainer.train()
print("\n✅ Training complete!")
print(f"Training loss: {train_result.metrics['train_loss']:.4f}")

## 9. Evaluate on Test Set

In [ ]:
# Validation results
val_metrics = trainer.evaluate(eval_dataset=encoded_dataset["validation"])
print("Validation Results:")
print(f"  Micro F1: {val_metrics['eval_micro_f1']:.4f}")
print(f"  Macro F1: {val_metrics['eval_macro_f1']:.4f}")

# Test results
test_metrics = trainer.evaluate(eval_dataset=encoded_dataset["test"], metric_key_prefix="test")
print(f"\nTest Results:")
print(f"  Micro F1: {test_metrics['test_micro_f1']:.4f}")
print(f"  Macro F1: {test_metrics['test_macro_f1']:.4f}")

## 10. Save the Trained Model

In [ ]:
import json
from pathlib import Path

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

# Save model + tokenizer
trainer.save_model(str(output_dir))
tokenizer.save_pretrained(str(output_dir))

# Save label map (needed by our app to map indices → emotion names)
label_map_path = output_dir / "label_map.json"
label_map_path.write_text(json.dumps({"labels": LABELS}, indent=2))

# Save training summary
summary = {
    "model_name": MODEL_NAME,
    "num_labels": NUM_LABELS,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "max_length": MAX_LENGTH,
    "validation_micro_f1": val_metrics["eval_micro_f1"],
    "validation_macro_f1": val_metrics["eval_macro_f1"],
    "test_micro_f1": test_metrics["test_micro_f1"],
    "test_macro_f1": test_metrics["test_macro_f1"],
}
(output_dir / "training_summary.json").write_text(json.dumps(summary, indent=2))

print(f"✅ Model saved to: {output_dir}")
print(f"\nFiles saved:")
for f in sorted(output_dir.iterdir()):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name:40s} {size_mb:.1f} MB")

## 11. Quick Sanity Check

Let's verify the saved model works by running a quick prediction.

In [ ]:
from transformers import pipeline

# Load the saved model fresh (proves the files are valid)
classifier = pipeline(
    "text-classification",
    model=str(output_dir),
    tokenizer=str(output_dir),
    top_k=5,
    function_to_apply="sigmoid",
    device=0 if torch.cuda.is_available() else -1,
)

test_texts = [
    "I'm feeling really sad and hopeless today",
    "Just got promoted at work! So excited!",
    "I'm so angry about what happened",
    "I feel anxious about the future",
    "Thank you so much for your help",
]

print("🧪 Sanity check predictions:\n")
for text in test_texts:
    result = classifier(text)
    top = result[0][:3]  # top 3 emotions
    emotions_str = ", ".join(f"{e['label']}={e['score']:.3f}" for e in top)
    print(f"  \"{text}\"")
    print(f"  → {emotions_str}\n")

## 12. Download the Model 📦

Run this cell to zip the model and download it.  
Then extract the zip into your project at: `models/bert_emotion/best_model/`

In [ ]:
import shutil

zip_path = shutil.make_archive("bert_emotion_model", "zip", ".", OUTPUT_DIR)
print(f"✅ Created: {zip_path}")

# Auto-download in Colab
try:
    from google.colab import files
    files.download(zip_path)
    print("📥 Download started!")
except ImportError:
    print(f"Not running in Colab. Manually download: {zip_path}")

## ✅ Done!

### Next steps:
1. Download the `bert_emotion_model.zip` file
2. Extract it into your MindGuard project:
   ```
   MindGuard/
   └── models/
       └── bert_emotion/
           └── best_model/
               ├── config.json
               ├── model.safetensors
               ├── tokenizer.json
               ├── tokenizer_config.json
               ├── label_map.json
               ├── training_summary.json
               └── ...
   ```
3. The updated `bert_classifier.py` in your project will automatically detect and load the model
4. Restart your API server: `uvicorn app.main:app --reload`